# Pipeline de datos masivos · Contratación pública en Antioquia (SECOP II)
**Gestión de Datos Masivos G002 — Institución Universitaria Pascual Bravo — 2026-2**
Integrantes: *Juan Manuel Tejada Bustamante*, *Diego Alejandro Castillo Perez* *Alejandra Munera*·

**Pregunta:** ¿Qué entidades públicas concentran el valor contratado en Antioquia y qué peso tiene la contratación directa, mes a mes?
**Usuarios:** veedurías ciudadanas, periodistas de datos y entes de control (Contraloría, Personerías).

### Cómo ejecutar
`Entorno de ejecución → Ejecutar todas`. La primera vez tarda unos 15–20 minutos: instala Java, descarga Hadoop y Kafka y luego corre el pipeline completo dos veces.
Todo corre dentro de esta máquina de Colab:
- **HDFS** en modo pseudo-distribuido (1 NameNode + 1 DataNode)
- **Kafka** en modo KRaft, sin Zookeeper
- **Spark** en modo local

Al final se descarga un ZIP con las evidencias, los datos de servicio y el tablero.

## Fase 1 · Fuente, pregunta y arquitectura
| Aspecto | Detalle |
|---|---|
| Fuente | SECOP II – Contratos Electrónicos (Colombia Compra Eficiente), API Socrata de **datos.gov.co** |
| Endpoint | `https://www.datos.gov.co/resource/jbjy-vk9h.json` (público, sin clave) |
| Filtro | `departamento = 'Antioquia'`; el rango de fechas se ajusta automáticamente a entre 20.000 y 100.000 contratos |
| Actualización | Diaria (se publican contratos nuevos todos los días) |
| **id_registro** | `id_contrato` (ej. `CO1.PCCNTR.4515200`), único por contrato |
| Columna de fecha | `fecha_de_firma`: tiempo del evento y marca de agua |

```
 API Socrata ──requests (paginación+reintentos)──▶ RAW local (JSONL con sobre, fecha_ingesta=…, manifiesto)
                                                     │ productor (key = id_registro)
                                                     ▼
                                     Kafka  tópico secop_contratos (3 particiones)
                     consumidor (grupo) │                         │ Spark Structured Streaming
                                        ▼                         ▼ (ventana 1 día, watermark, checkpoint)
           HDFS RAW /datalake/raw/…/fecha_ingesta=…        CAPA RÁPIDA (ventanas diarias)
              │ MapReduce (Hadoop Streaming)                      │
              │ PySpark: limpieza + calidad + dedup               │
              ▼                                                   │
           HDFS CURADA Parquet particionado anio_mes=…            │
              └───────────────▶  SERVICIO SQLite (indicadores) ◀──┘
                                        ▼
                                 Tablero Plotly (HTML)
```
**¿Por qué Lambda y no Kappa?** La pregunta es histórica (meses completos, ranking de entidades). Para responderla hace falta la limpieza completa: deduplicar contra todo lo ya curado, corregir tipos y validar la calidad antes de publicar, y eso es más natural y verificable en batch. Como SECOP publica a diario, una capa rápida da conteos diarios sin esperar al batch. Las dos capas se unen en la tabla de servicio `ind_serie_diaria`: el batch cubre hasta su fecha de corte y el streaming los días posteriores. Con Kappa habría que reprocesar todo el histórico desde Kafka cada vez que cambie la lógica de limpieza.

In [1]:
# ===================== PARÁMETROS DEL PROYECTO =====================
from pathlib import Path

BASE = Path("/content/pipeline-secop-antioquia")
FUENTE, DATASET_ID, DATASET_NOMBRE = "datos.gov.co", "jbjy-vk9h", "secop_contratos"
API_URL = f"https://www.datos.gov.co/resource/{DATASET_ID}.json"
DEPARTAMENTO = "Antioquia"
ID_REGISTRO, CAMPO_FECHA = "id_contrato", "fecha_de_firma"
PAGE_SIZE = 5000
OBJETIVO_MAX_REGISTROS = 100_000   # la primera carga elige una fecha inicial para no pasar de aquí
MIN_REGISTROS = 20_000             # mínimo exigido por el enunciado
FECHAS_CANDIDATAS = ["2025-07-01", "2025-10-01", "2026-01-01", "2026-03-01", "2026-05-01",
                     "2026-07-01", "2026-08-01", "2026-09-01", "2026-10-01"]

# Columnas pedidas a la API. Se excluyen a propósito los datos personales que no aportan
# a la pregunta (cuentas bancarias, documentos de representantes legales, supervisores y
# ordenadores del gasto): principio de minimización de datos.
COLUMNAS_API = [
    "id_contrato", "referencia_del_contrato", "proceso_de_compra",
    "nombre_entidad", "nit_entidad", "codigo_entidad",
    "departamento", "ciudad", "orden", "sector", "rama", "entidad_centralizada",
    "estado_contrato", "codigo_de_categoria_principal",
    "tipo_de_contrato", "modalidad_de_contratacion", "justificacion_modalidad_de",
    "fecha_de_firma", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato",
    "tipodocproveedor", "codigo_proveedor", "es_pyme",
    "origen_de_los_recursos", "destino_gasto",
    "valor_del_contrato", "valor_facturado", "valor_pagado", "dias_adicionados",
    "duraci_n_del_contrato", "objeto_del_contrato", "ultima_actualizacion",
]

KAFKA_BOOTSTRAP, KAFKA_TOPIC, KAFKA_GRUPO = "localhost:9092", "secop_contratos", "secop-hdfs-writer"
KAFKA_PARTICIONES = 3
HDFS_URI = "hdfs://localhost:9000"
HDFS_RAW = f"/datalake/raw/{FUENTE}/{DATASET_NOMBRE}"
HDFS_CURADA = f"/datalake/curada/{FUENTE}/{DATASET_NOMBRE}"
HDFS_STAGING = f"/datalake/tmp/{DATASET_NOMBRE}_staging"
HDFS_MR_OUT = "/datalake/mapreduce/conteo_por_modalidad"
HDFS_CHECKPOINT = "/checkpoints/secop_stream_diario"

RAW_DIR = BASE / "data" / "raw"
MANIFEST_DIR = BASE / "data" / "manifiestos"
MUESTRAS_DIR = BASE / "data" / "muestras"
SERVICIO_DIR = BASE / "data" / "servicio"
ESTADO_DIR = BASE / "estado"
EVID = BASE / "evidencias"
DASH = BASE / "dashboard"
MR_DIR = BASE / "mapreduce"
SQLITE_PATH = SERVICIO_DIR / "secop_servicio.db"
WATERMARK_FILE = ESTADO_DIR / "watermark.json"
PRODUCTOR_ESTADO = ESTADO_DIR / "productor_archivos_publicados.json"
CURADA_ESTADO = ESTADO_DIR / "curada_archivos_procesados.json"
for d in [RAW_DIR, MANIFEST_DIR, MUESTRAS_DIR, SERVICIO_DIR, ESTADO_DIR, EVID, EVID / "calidad", DASH, MR_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Proyecto en", BASE)

Proyecto en /content/pipeline-secop-antioquia


## Instalación del entorno (Java, Hadoop, Kafka, PySpark)
Se instalan Java 11 (para Hadoop) y Java 17 (para Spark y Kafka). Hadoop 3.3.6 y Kafka 3.7.0 se descargan de los servidores oficiales de Apache. Este paso tarda unos 5 minutos.

In [2]:
import os, sys, subprocess, time, glob, shutil, socket

def sh(cmd, check=True, quiet=False, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    if not quiet and r.stdout.strip():
        print(r.stdout[-4000:])
    if check and r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-4000:])
        raise RuntimeError(f"Falló el comando: {cmd}")
    return r

t0 = time.time()
sh("apt-get -qq update", quiet=True)
sh("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y openjdk-11-jdk-headless openjdk-17-jdk-headless", quiet=True)
sh(f"{sys.executable} -m pip -q install 'kafka-python>=2.0.3' plotly requests", quiet=True)
try:
    import pyspark
except ImportError:
    sh(f"{sys.executable} -m pip -q install 'pyspark>=3.5,<3.6'", quiet=True)
    import pyspark
print("Python", sys.version.split()[0], "| PySpark", pyspark.__version__)

HADOOP_VER, KAFKA_VER = "3.3.6", "3.7.0"

def descargar(urls, destino):
    for u in urls:
        print("Descargando", u)
        r = subprocess.run(["wget", "-q", "-O", destino, u])
        if r.returncode == 0 and os.path.getsize(destino) > 10_000_000:
            return
    raise RuntimeError(f"No se pudo descargar {destino}")

if not Path("/content/hadoop").exists():
    descargar([f"https://dlcdn.apache.org/hadoop/common/hadoop-{HADOOP_VER}/hadoop-{HADOOP_VER}.tar.gz",
               f"https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VER}/hadoop-{HADOOP_VER}.tar.gz"],
              "/content/hadoop.tgz")
    sh("tar -xzf /content/hadoop.tgz -C /content && mv /content/hadoop-* /content/hadoop && rm /content/hadoop.tgz")
if not Path("/content/kafka").exists():
    descargar([f"https://archive.apache.org/dist/kafka/{KAFKA_VER}/kafka_2.13-{KAFKA_VER}.tgz",
               f"https://dlcdn.apache.org/kafka/{KAFKA_VER}/kafka_2.13-{KAFKA_VER}.tgz"],
              "/content/kafka.tgz")
    sh("tar -xzf /content/kafka.tgz -C /content && mv /content/kafka_2.13-* /content/kafka && rm /content/kafka.tgz")

JAVA11 = "/usr/lib/jvm/java-11-openjdk-amd64"
JAVA17 = "/usr/lib/jvm/java-17-openjdk-amd64"
assert Path(JAVA11).exists() and Path(JAVA17).exists(), "No se instaló Java"
os.environ["JAVA_HOME"] = JAVA17
os.environ["HADOOP_HOME"] = "/content/hadoop"
os.environ["HADOOP_CONF_DIR"] = "/content/hadoop/etc/hadoop"
os.environ["PATH"] = "/content/hadoop/bin:/content/hadoop/sbin:/content/kafka/bin:" + os.environ["PATH"]
print(f"Instalación lista en {time.time() - t0:.0f} s")

Python 3.13.18 | PySpark 4.0.5
Descargando https://dlcdn.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
Descargando https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
Instalación lista en 122 s


### Arranque de HDFS y Kafka
- **HDFS** queda en modo pseudo-distribuido, con `fs.defaultFS = hdfs://localhost:9000`, replicación 1 y permisos desactivados. Interfaz web del NameNode: puerto 9870.
- **Kafka** corre en modo KRaft (broker y controller en un mismo proceso) en el puerto 9092.

In [4]:
conf_dir = Path("/content/hadoop/etc/hadoop")

def xml(props):
    filas = "".join(f"  <property><name>{k}</name><value>{v}</value></property>\n" for k, v in props.items())
    return f'<?xml version="1.0"?>\n<configuration>\n{filas}</configuration>\n'

(conf_dir / "core-site.xml").write_text(xml({
    "fs.defaultFS": HDFS_URI, "hadoop.tmp.dir": "/content/hdfs/tmp"}))
(conf_dir / "hdfs-site.xml").write_text(xml({
    "dfs.replication": "1",
    "dfs.namenode.name.dir": "file:///content/hdfs/nn",
    "dfs.datanode.data.dir": "file:///content/hdfs/dn",
    "dfs.permissions.enabled": "false",
    "dfs.webhdfs.enabled": "true"}))
env_sh = conf_dir / "hadoop-env.sh"
if "# pipeline-secop" not in env_sh.read_text():
    with open(env_sh, "a") as f:
        f.write(f"\n# pipeline-secop\nexport JAVA_HOME={JAVA11}\nexport HDFS_NAMENODE_USER=root\n"
                "export HDFS_DATANODE_USER=root\nexport HDFS_SECONDARYNAMENODE_USER=root\n"
                "export HADOOP_HEAPSIZE_MAX=1g\n")

def puerto_abierto(p):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("localhost", p)) == 0

# ---- HDFS ----
if not puerto_abierto(9870):
    if not Path("/content/hdfs/nn/current").exists():
        sh("hdfs namenode -format -force -nonInteractive", quiet=True)
    sh("hdfs --daemon start namenode")
    sh("hdfs --daemon start datanode")
sh("hdfs dfsadmin -safemode wait", quiet=True)
for _ in range(60):
    rep = sh("hdfs dfsadmin -report", quiet=True, check=False).stdout
    if "Live datanodes (1)" in rep:
        break
    time.sleep(2)
print([l for l in rep.splitlines() if "Live datanodes" in l or "Configured Capacity" in l])
sh("hdfs dfs -mkdir -p /datalake /checkpoints")

# ---- Kafka (KRaft) ----
if not puerto_abierto(9092):
    props = "/content/kafka/config/kraft/server.properties"
    if not Path("/tmp/kraft-combined-logs/meta.properties").exists():
        cluster_id = sh("kafka-storage.sh random-uuid", quiet=True).stdout.strip().splitlines()[-1]
        sh(f"kafka-storage.sh format -t {cluster_id} -c {props}", quiet=True)
    subprocess.Popen(["/content/kafka/bin/kafka-server-start.sh", props],
                     stdout=open("/content/kafka.log", "w"), stderr=subprocess.STDOUT,
                     env=dict(os.environ, KAFKA_HEAP_OPTS="-Xmx768M -Xms512M"), start_new_session=True)
    for _ in range(60):
        if puerto_abierto(9092):
            break
        time.sleep(1)
    time.sleep(5)
print("Kafka escuchando en 9092:", puerto_abierto(9092))
print("Procesos Java:", sh(f"{JAVA17}/bin/jps", quiet=True, check=False).stdout.split())

['Configured Capacity: 115658190848 (107.72 GB)', 'Live datanodes (1):', 'Configured Capacity: 115658190848 (107.72 GB)']
Kafka escuchando en 9092: True
Procesos Java: ['13922', 'DataNode', '16163', 'Jps', '15636', 'Kafka', '13862', 'NameNode']


### Sesión de Spark y utilidades
Spark corre en modo local y lee y escribe en el HDFS anterior. El conector de Kafka para Structured Streaming se descarga de Maven al crear la sesión. Las operaciones de archivos en HDFS (listar, escribir lotes, medir tamaños) se hacen con la API `FileSystem` de Hadoop a través de Spark.

In [5]:
import json, csv, uuid, sqlite3
from datetime import datetime, timezone
from collections import Counter, defaultdict
import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

_scala = "2.12" if pyspark.__version__.startswith("3") else "2.13"
spark = (SparkSession.builder.appName("pipeline_secop").master("local[*]")
         .config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_{_scala}:{pyspark.__version__}")
         .config("spark.hadoop.fs.defaultFS", HDFS_URI)
         .config("spark.sql.session.timeZone", "America/Bogota")
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.ansi.enabled", "false")      # cast inválido -> nulo (se cuenta y se decide)
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

# ---------- utilidades ----------
def ahora_iso():
    return datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")

def leer_json(path, default):
    return json.loads(Path(path).read_text(encoding="utf-8")) if Path(path).exists() else default

def escribir_json(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)

def titulo(t):
    print("\n" + "=" * 72 + "\n" + t + "\n" + "=" * 72)

# ---------- HDFS a través de la API FileSystem de Hadoop ----------
_jvm = spark._jvm
_FS = _jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
_P = _jvm.org.apache.hadoop.fs.Path

def hdfs_existe(p):
    return _FS.exists(_P(p))

def hdfs_escribir(p, texto):
    out = _FS.create(_P(p), False)
    out.write(bytearray(texto.encode("utf-8")))
    out.close()

def hdfs_listar(p, ext=".jsonl"):
    if not hdfs_existe(p):
        return []
    it, res = _FS.listFiles(_P(p), True), []
    while it.hasNext():
        ruta = it.next().getPath().toUri().getPath()
        if ruta.endswith(ext):
            res.append(ruta)
    return sorted(res)

def hdfs_tamano(p):
    return _FS.getContentSummary(_P(p)).getLength() if hdfs_existe(p) else 0

def hdfs_borrar(p):
    if hdfs_existe(p):
        _FS.delete(_P(p), True)

# ---------- esquema explícito del "sobre" JSON ----------
# Socrata devuelve todo como texto: se declara el esquema (sin inferir) y los tipos se
# corrigen de forma explícita en la zona curada.
PAYLOAD = StructType([StructField(c, StringType(), True) for c in COLUMNAS_API])
SOBRE = StructType([StructField(c, StringType(), True) for c in
                    ["fuente", "dataset", "id_registro", "fecha_ingesta", "run_id"]]
                   + [StructField("payload", PAYLOAD, True)])
SOBRE_CON_CORRUPTOS = StructType(SOBRE.fields + [StructField("_corrupt_record", StringType(), True)])

def leer_raw(rutas=None):
    lector = spark.read.schema(SOBRE_CON_CORRUPTOS).option("mode", "PERMISSIVE")
    if rutas:
        return lector.json([f"{HDFS_URI}{r}" for r in rutas])
    return lector.json(f"{HDFS_URI}{HDFS_RAW}/*/*.jsonl")   # glob: la fecha_ingesta ya va dentro del sobre

Spark 4.0.5


## Fase 2 · Ingesta a la zona raw
- **Paginación:** `$limit = 5000` y `$offset`, con un orden estable (`fecha_de_firma, id_contrato`). Sin un orden fijo, Socrata puede repetir u omitir filas entre páginas.
- **Reintentos:** 5 intentos con espera exponencial (2, 4, 8, 16 s) ante errores de red, timeouts, HTTP 429 y 5xx. Un 4xx distinto de 429 es un error de la consulta y no se reintenta.
- **Sobre JSON:** `{fuente, dataset, id_registro, fecha_ingesta, run_id, payload}`. El `payload` es el registro tal como lo devolvió la API, **sin modificar**.
- **Archivos:** JSON Lines en `data/raw/datos.gov.co/secop_contratos/fecha_ingesta=AAAA-MM-DD/part-<run_id>.jsonl`.
- **Manifiesto** por ejecución, con URL, parámetros, páginas, conteo, reintentos y marcas de agua.
- **Marca de agua** (`estado/watermark.json`): guarda la última `fecha_de_firma` cargada. La siguiente corrida solo pide `fecha_de_firma >= marca`. Se usa `>=` para no perder contratos firmados en el mismo instante; ese traslape se elimina después con el dedup por `id_registro`.
- **Primera carga:** se cuenta en la API cuántos contratos hay desde cada fecha candidata y se elige la primera que deje entre 20.000 y 100.000.

In [6]:
import requests

INTENTOS = 5

def pedir(session, params, stats):
    for intento in range(1, INTENTOS + 1):
        try:
            r = session.get(API_URL, params=params, timeout=120)
            if r.status_code == 200:
                return r.json()
            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()                      # error de la consulta: no se reintenta
            raise requests.ConnectionError(f"HTTP {r.status_code}")
        except (requests.ConnectionError, requests.Timeout, ValueError) as e:
            stats["reintentos"] += 1
            if intento == INTENTOS:
                raise
            espera = 2 ** intento
            print(f"   [reintento {intento}/{INTENTOS}] {e} -> espero {espera}s")
            time.sleep(espera)

def contar_api(desde, hasta):
    where = f"departamento = '{DEPARTAMENTO}' AND {CAMPO_FECHA} >= '{desde}' AND {CAMPO_FECHA} <= '{hasta}'"
    res = pedir(requests.Session(), {"$select": "count(*) AS n", "$where": where}, {"reintentos": 0})
    return int(res[0]["n"])

def elegir_fecha_inicial(hasta):
    print("Conteo en la API por fecha inicial candidata:")
    elegida = FECHAS_CANDIDATAS[-1]
    for f in FECHAS_CANDIDATAS:
        n = contar_api(f + "T00:00:00.000", hasta)
        print(f"   desde {f}: {n:,} contratos")
        if n <= OBJETIVO_MAX_REGISTROS:
            elegida = f
            if n < MIN_REGISTROS and f != FECHAS_CANDIDATAS[0]:
                print("   (por debajo del mínimo: se usa la fecha anterior)")
                elegida = FECHAS_CANDIDATAS[FECHAS_CANDIDATAS.index(f) - 1]
            break
    print(f"Fecha inicial elegida: {elegida}")
    return elegida + "T00:00:00.000"

def ingestar():
    titulo("FASE 2 - INGESTA API SECOP II -> ZONA RAW")
    inicio = time.time()
    run_id = datetime.now().strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
    fecha_ingesta_ts = ahora_iso()
    hasta = datetime.now().strftime("%Y-%m-%dT23:59:59.999")   # evita fechas futuras mal digitadas
    estado = leer_json(WATERMARK_FILE, {})
    wm_anterior = estado.get("ultima_fecha_de_firma")
    desde = wm_anterior or elegir_fecha_inicial(hasta)

    where = (f"departamento = '{DEPARTAMENTO}' AND {CAMPO_FECHA} >= '{desde}' "
             f"AND {CAMPO_FECHA} <= '{hasta}'")
    params_base = {"$select": ",".join(COLUMNAS_API), "$where": where,
                   "$order": f"{CAMPO_FECHA} ASC, {ID_REGISTRO} ASC", "$limit": PAGE_SIZE}
    print(f"run_id: {run_id} | marca de agua anterior: {wm_anterior}\nConsulta: {where}")

    carpeta = RAW_DIR / FUENTE / DATASET_NOMBRE / f"fecha_ingesta={fecha_ingesta_ts[:10]}"
    carpeta.mkdir(parents=True, exist_ok=True)
    archivo = carpeta / f"part-{run_id}.jsonl"
    session = requests.Session()
    stats = {"reintentos": 0}
    total, paginas, sin_id, offset, max_fecha = 0, 0, 0, 0, wm_anterior
    with open(archivo, "w", encoding="utf-8") as f:
        while True:
            registros = pedir(session, dict(params_base, **{"$offset": offset}), stats)
            if not registros:
                break
            paginas += 1
            for reg in registros:
                id_reg = reg.get(ID_REGISTRO)
                sin_id += 0 if id_reg else 1
                f.write(json.dumps({"fuente": FUENTE, "dataset": DATASET_ID, "id_registro": id_reg,
                                    "fecha_ingesta": fecha_ingesta_ts, "run_id": run_id,
                                    "payload": reg}, ensure_ascii=False) + "\n")
                fe = reg.get(CAMPO_FECHA)
                if fe and (max_fecha is None or fe > max_fecha):
                    max_fecha = fe
            total += len(registros)
            print(f"   página {paginas:>3} | offset {offset:>7} | +{len(registros):>5} | acumulado {total:>7,}")
            if len(registros) < PAGE_SIZE:
                break
            offset += PAGE_SIZE

    if total == 0:
        archivo.unlink(); archivo_rel = None; estado_run = "sin_datos_nuevos"
    else:
        archivo_rel = str(archivo.relative_to(BASE)); estado_run = "ok"
        escribir_json(WATERMARK_FILE, {"ultima_fecha_de_firma": max_fecha, "actualizado": ahora_iso(), "run_id": run_id})
        with open(archivo, encoding="utf-8") as src, open(MUESTRAS_DIR / "raw_muestra.jsonl", "w", encoding="utf-8") as dst:
            for i, linea in enumerate(src):
                if i >= 500:
                    break
                dst.write(linea)
    manifiesto = {
        "run_id": run_id, "fuente": FUENTE, "dataset": DATASET_ID, "url": API_URL,
        "parametros": params_base,
        "paginacion": {"tipo": "$limit/$offset", "tamano_pagina": PAGE_SIZE, "paginas": paginas},
        "conteo_registros": total, "registros_sin_id": sin_id, "reintentos": stats["reintentos"],
        "archivo": archivo_rel, "fecha_ingesta": fecha_ingesta_ts,
        "watermark_anterior": wm_anterior, "watermark_nuevo": max_fecha,
        "duracion_segundos": round(time.time() - inicio, 1), "estado": estado_run,
    }
    escribir_json(MANIFEST_DIR / f"manifiesto_{run_id}.json", manifiesto)
    print(f"\nRegistros: {total:,} (sin id: {sin_id}, reintentos: {stats['reintentos']}) | nueva marca de agua: {max_fecha}")
    return manifiesto

manifiesto_1 = ingestar()
print(json.dumps(manifiesto_1, indent=2, ensure_ascii=False))


FASE 2 - INGESTA API SECOP II -> ZONA RAW
Conteo en la API por fecha inicial candidata:
   desde 2025-07-01: 149,893 contratos
   desde 2025-10-01: 123,752 contratos
   desde 2026-01-01: 106,285 contratos
   desde 2026-03-01: 55,963 contratos
Fecha inicial elegida: 2026-03-01
run_id: 20261124T214438_4a6257 | marca de agua anterior: None
Consulta: departamento = 'Antioquia' AND fecha_de_firma >= '2026-03-01T00:00:00.000' AND fecha_de_firma <= '2026-11-24T23:59:59.999'
   página   1 | offset       0 | + 5000 | acumulado   5,000
   página   2 | offset    5000 | + 5000 | acumulado  10,000
   página   3 | offset   10000 | + 5000 | acumulado  15,000
   página   4 | offset   15000 | + 5000 | acumulado  20,000
   página   5 | offset   20000 | + 5000 | acumulado  25,000
   página   6 | offset   25000 | + 5000 | acumulado  30,000
   página   7 | offset   30000 | + 5000 | acumulado  35,000
   página   8 | offset   35000 | + 5000 | acumulado  40,000
   página   9 | offset   40000 | + 5000 | acumu

In [7]:
# Un sobre de ejemplo de la zona raw (el payload es el registro original de la API)
print(json.dumps(json.loads(open(MUESTRAS_DIR / "raw_muestra.jsonl", encoding="utf-8").readline()), indent=2, ensure_ascii=False)[:2500])

{
  "fuente": "datos.gov.co",
  "dataset": "jbjy-vk9h",
  "id_registro": "CO1.PCCNTR.9366213",
  "fecha_ingesta": "2026-11-24T21:44:38+00:00",
  "run_id": "20261124T214438_4a6257",
  "payload": {
    "id_contrato": "CO1.PCCNTR.9366213",
    "referencia_del_contrato": "CO1.PCCNTR.9366213-",
    "proceso_de_compra": "CO1.BDOS.10050613",
    "nombre_entidad": "EMPRESA SOCIAL DEL ESTADO HOSPITAL SAN VICENTE DE PAUL",
    "nit_entidad": "890905193",
    "codigo_entidad": "705639300",
    "departamento": "Antioquia",
    "ciudad": "Barbosa",
    "orden": "Territorial",
    "sector": "Salud y Protección Social",
    "rama": "Corporación Autónoma",
    "entidad_centralizada": "Centralizada",
    "estado_contrato": "En ejecución",
    "codigo_de_categoria_principal": "V1.80161500",
    "tipo_de_contrato": "Decreto 092 de 2017",
    "modalidad_de_contratacion": "Contratación régimen especial",
    "justificacion_modalidad_de": "Decreto 092 de 2017",
    "fecha_de_firma": "2026-03-02T08:35:35.729

## Fase 3 · Mensajería con Kafka
- El tópico `secop_contratos` tiene **3 particiones**. El productor publica cada línea del raw con **key = id_registro**: el mismo contrato siempre cae en la misma partición y conserva su orden.
- El productor usa `acks='all'`, reintentos y `max_in_flight_requests_per_connection=1`, para que un reintento de red no pierda ni desordene mensajes.
- **¿Qué pasa si el productor se ejecuta dos veces?** Kafka **no** deduplica por key, así que cada contrato quedaría dos veces en el tópico. Lo evitamos en tres niveles:
  1. El productor lleva un registro (`estado/productor_archivos_publicados.json`) y solo publica archivos raw que no ha publicado antes. La celda siguiente lo corre dos veces para demostrarlo.
  2. El consumidor continúa desde su offset confirmado, sin releer lo que ya escribió.
  3. Si igual entra un duplicado (por el traslape de la marca de agua o por un reenvío), la zona curada **deduplica por `id_registro`**.

In [8]:
from kafka import KafkaAdminClient, KafkaConsumer, KafkaProducer
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

def asegurar_topico():
    for intento in range(10):
        try:
            admin = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP, client_id="secop-admin")
            break
        except Exception as e:
            print("Esperando a Kafka...", e); time.sleep(3)
    try:
        admin.create_topics([NewTopic(name=KAFKA_TOPIC, num_partitions=KAFKA_PARTICIONES, replication_factor=1)])
        print(f"Tópico creado: {KAFKA_TOPIC}")
    except TopicAlreadyExistsError:
        print(f"Tópico ya existe: {KAFKA_TOPIC}")
    admin.close()
    c = KafkaConsumer(bootstrap_servers=KAFKA_BOOTSTRAP)
    n = len(c.partitions_for_topic(KAFKA_TOPIC) or [])
    c.close()
    print(f"Particiones del tópico: {n}")
    return n

def producir(forzar=False):
    titulo("FASE 3 - PRODUCTOR KAFKA (raw -> tópico, key = id_registro)")
    n_part = asegurar_topico()
    archivos = sorted((RAW_DIR / FUENTE / DATASET_NOMBRE).glob("fecha_ingesta=*/*.jsonl"))
    publicados = leer_json(PRODUCTOR_ESTADO, {})
    pendientes = [a for a in archivos if forzar or str(a.relative_to(BASE)) not in publicados]
    print(f"Archivos raw: {len(archivos)} | ya publicados: {len(publicados)} | a publicar: {len(pendientes)}")
    if not pendientes:
        print("Nada nuevo que publicar: el productor no genera duplicados al correr de nuevo.")
        return 0
    producer = KafkaProducer(bootstrap_servers=KAFKA_BOOTSTRAP,
                             key_serializer=lambda k: k.encode("utf-8"),
                             value_serializer=lambda v: v.encode("utf-8"),
                             acks="all", retries=5, max_in_flight_requests_per_connection=1,
                             linger_ms=50, batch_size=256 * 1024, compression_type="gzip")
    por_particion = Counter()
    total = 0
    for archivo in pendientes:
        n = 0
        with open(archivo, encoding="utf-8") as f:
            for linea in f:
                linea = linea.rstrip("\n")
                if not linea:
                    continue
                key = json.loads(linea).get("id_registro") or "SIN_ID"
                producer.send(KAFKA_TOPIC, key=key, value=linea).add_callback(
                    lambda md: por_particion.update([md.partition]))
                n += 1
        producer.flush()
        total += n
        publicados[str(archivo.relative_to(BASE))] = {"mensajes": n, "publicado": ahora_iso()}
        escribir_json(PRODUCTOR_ESTADO, publicados)
        print(f"   {archivo.name}: {n:,} mensajes")
    producer.close()
    resumen = {"fecha": ahora_iso(), "topico": KAFKA_TOPIC, "particiones_topico": n_part,
               "mensajes_publicados": total, "mensajes_por_particion": dict(sorted(por_particion.items()))}
    escribir_json(EVID / "kafka_productor_ultimo.json", resumen)
    print(f"Total publicados: {total:,} | por partición: {resumen['mensajes_por_particion']}")
    return total

producir()
producir()   # segunda ejecución: no vuelve a publicar los mismos archivos


FASE 3 - PRODUCTOR KAFKA (raw -> tópico, key = id_registro)
Tópico creado: secop_contratos


/tmp/ipykernel_12638/1784520652.py:64: DeprecationWarning: key_serializer does not implement kafka.serializer.Serializer
  producir()
/tmp/ipykernel_12638/1784520652.py:64: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producir()


Particiones del tópico: 3
Archivos raw: 1 | ya publicados: 0 | a publicar: 1


   part-20261124T214438_4a6257.jsonl: 55,963 mensajes
Total publicados: 55,963 | por partición: {0: 18752, 1: 18611, 2: 18600}

FASE 3 - PRODUCTOR KAFKA (raw -> tópico, key = id_registro)
Tópico ya existe: secop_contratos
Particiones del tópico: 3
Archivos raw: 1 | ya publicados: 1 | a publicar: 0
Nada nuevo que publicar: el productor no genera duplicados al correr de nuevo.


0

## Fase 4 · Data lake en HDFS y MapReduce
El consumidor lee el tópico con el grupo `secop-hdfs-writer` y escribe en HDFS:
`/datalake/raw/datos.gov.co/secop_contratos/fecha_ingesta=AAAA-MM-DD/lote_….jsonl`
- **Un archivo por lote** de hasta 10.000 mensajes, no uno por mensaje: HDFS guarda los metadatos de cada archivo en la memoria del NameNode, y millones de archivos pequeños la saturan.
- `enable_auto_commit=False`: el offset se confirma **solo después** de escribir el lote en HDFS. Si el proceso cae antes, el lote se relee (entrega "al menos una vez") y el duplicado se elimina en la zona curada.

In [9]:
TAMANO_LOTE = 10_000

def consumir_a_hdfs(inactividad=15):
    titulo("FASE 3/4 - CONSUMIDOR KAFKA -> HDFS (un archivo por lote)")
    consumer = KafkaConsumer(KAFKA_TOPIC, bootstrap_servers=KAFKA_BOOTSTRAP, group_id=KAFKA_GRUPO,
                             enable_auto_commit=False, auto_offset_reset="earliest",
                             max_poll_records=5000, value_deserializer=lambda b: b.decode("utf-8"))
    buffer, archivos, por_particion = [], [], Counter()
    n_lote, total, ultimo = 0, 0, time.time()
    while True:
        registros = consumer.poll(timeout_ms=2000)
        for tp, mensajes in registros.items():
            por_particion[tp.partition] += len(mensajes)
            buffer.extend(m.value for m in mensajes)
        if registros:
            ultimo = time.time()
        inactivo = time.time() - ultimo > inactividad
        if buffer and (len(buffer) >= TAMANO_LOTE or inactivo or not registros):
            n_lote += 1
            grupos = defaultdict(list)
            for v in buffer:
                try:
                    grupos[json.loads(v).get("fecha_ingesta", "")[:10] or "desconocida"].append(v)
                except json.JSONDecodeError:
                    grupos["corruptos"].append(v)
            sello = datetime.now().strftime("%Y%m%dT%H%M%S")
            for fecha, lineas in grupos.items():
                ruta = f"{HDFS_RAW}/fecha_ingesta={fecha}/lote_{sello}_{n_lote:04d}_{uuid.uuid4().hex[:6]}.jsonl"
                hdfs_escribir(ruta, "\n".join(lineas) + "\n")
                archivos.append({"ruta": ruta, "mensajes": len(lineas)})
                print(f"   lote {n_lote}: {len(lineas):>6,} mensajes -> {ruta}")
            consumer.commit()                      # confirmar offsets solo después de escribir
            total += len(buffer)
            buffer = []
        if inactivo:
            break
    consumer.close()
    resumen = {"fecha": ahora_iso(), "grupo": KAFKA_GRUPO, "mensajes_consumidos": total,
               "mensajes_por_particion": dict(sorted(por_particion.items())), "lotes": n_lote,
               "archivos_hdfs": archivos}
    escribir_json(EVID / "kafka_consumidor_ultimo.json", resumen)
    print(f"Mensajes consumidos: {total:,} en {n_lote} lote(s) | por partición: {resumen['mensajes_por_particion']}")
    return total

consumir_a_hdfs()
print(sh(f"hdfs dfs -ls -R {HDFS_RAW}", quiet=True).stdout)

/tmp/ipykernel_12638/1949313614.py:5: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(KAFKA_TOPIC, bootstrap_servers=KAFKA_BOOTSTRAP, group_id=KAFKA_GRUPO,



FASE 3/4 - CONSUMIDOR KAFKA -> HDFS (un archivo por lote)


   lote 1: 10,000 mensajes -> /datalake/raw/datos.gov.co/secop_contratos/fecha_ingesta=2026-11-24/lote_20261124T214850_0001_49e5f0.jsonl
   lote 2: 10,000 mensajes -> /datalake/raw/datos.gov.co/secop_contratos/fecha_ingesta=2026-11-24/lote_20261124T214853_0002_515246.jsonl
   lote 3: 10,000 mensajes -> /datalake/raw/datos.gov.co/secop_contratos/fecha_ingesta=2026-11-24/lote_20261124T214855_0003_8d9f30.jsonl
   lote 4: 10,000 mensajes -> /datalake/raw/datos.gov.co/secop_contratos/fecha_ingesta=2026-11-24/lote_20261124T214857_0004_878f8e.jsonl
   lote 5: 10,000 mensajes -> /datalake/raw/datos.gov.co/secop_contratos/fecha_ingesta=2026-11-24/lote_20261124T214859_0005_15ad4d.jsonl
   lote 6:  5,963 mensajes -> /datalake/raw/datos.gov.co/secop_contratos/fecha_ingesta=2026-11-24/lote_20261124T214902_0006_dee7ed.jsonl
Mensajes consumidos: 55,963 en 6 lote(s) | por partición: {0: 18752, 1: 18611, 2: 18600}
drwxr-xr-x   - root supergroup          0 2026-11-24 21:49 /datalake/raw/datos.gov.co/sec

### MapReduce con Hadoop Streaming (mapper y reducer en Python)
El job cuenta los contratos y suma el valor por **modalidad de contratación** sobre la zona raw de HDFS, sin deduplicar.
- Modalidad vacía → `SIN_DATO`; línea corrupta → `__CORRUPTO__`.
- Valor no numérico → 0; los valores se redondean al entero con HALF_UP.

In [10]:
(MR_DIR / "mapper.py").write_text(r"""#!/usr/bin/env python3
# MAPPER: sobre JSON -> modalidad \t 1 \t valor_entero
import io, json, sys
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
entrada = io.TextIOWrapper(sys.stdin.buffer, encoding="utf-8", errors="replace")
salida = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8")

def a_entero(valor):
    if valor is None:
        return 0
    try:
        return int(Decimal(str(valor).strip()).quantize(Decimal("1"), rounding=ROUND_HALF_UP))
    except (InvalidOperation, ValueError):
        return 0

for linea in entrada:
    linea = linea.strip()
    if not linea:
        continue
    try:
        sobre = json.loads(linea)
    except ValueError:
        salida.write("__CORRUPTO__\t1\t0\n")
        continue
    p = sobre.get("payload") or {}
    modalidad = (p.get("modalidad_de_contratacion") or "").strip().replace("\t", " ") or "SIN_DATO"
    salida.write("%s\t1\t%d\n" % (modalidad, a_entero(p.get("valor_del_contrato"))))
salida.flush()
""", encoding="utf-8")

(MR_DIR / "reducer.py").write_text(r"""#!/usr/bin/env python3
# REDUCER: Hadoop entrega las claves ordenadas -> se acumula mientras la clave no cambie
import io, sys
entrada = io.TextIOWrapper(sys.stdin.buffer, encoding="utf-8", errors="replace")
salida = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8")
actual, conteo, suma = None, 0, 0
for linea in entrada:
    partes = linea.rstrip("\n").split("\t")
    if len(partes) != 3:
        continue
    clave, n, valor = partes[0], int(partes[1]), int(partes[2])
    if clave != actual:
        if actual is not None:
            salida.write("%s\t%d\t%d\n" % (actual, conteo, suma))
        actual, conteo, suma = clave, 0, 0
    conteo += n
    suma += valor
if actual is not None:
    salida.write("%s\t%d\t%d\n" % (actual, conteo, suma))
salida.flush()
""", encoding="utf-8")

def correr_mapreduce():
    titulo("FASE 4 - MAPREDUCE (Hadoop Streaming)")
    jar = glob.glob("/content/hadoop/share/hadoop/tools/lib/hadoop-streaming-*.jar")[0]
    sh(f"hdfs dfs -rm -r -f {HDFS_MR_OUT}", quiet=True)
    r = sh(f'cd {MR_DIR} && hadoop jar {jar} -D mapreduce.job.name=secop_conteo_por_modalidad -D mapreduce.job.reduces=1 '
           f'-files {MR_DIR}/mapper.py,{MR_DIR}/reducer.py -cmdenv PYTHONIOENCODING=utf-8 '
           f'-mapper "python3 {MR_DIR}/mapper.py" -reducer "python3 {MR_DIR}/reducer.py" '
           f'-input "{HDFS_RAW}/*/*.jsonl" -output {HDFS_MR_OUT}', quiet=True)
    contadores = [l.strip() for l in r.stderr.splitlines()
                  if any(k in l for k in ("Map input records", "Map output records", "Reduce output records", "completed successfully"))]
    print("\n".join(contadores))
    resultado = sh(f"hdfs dfs -cat {HDFS_MR_OUT}/part-*", quiet=True).stdout
    (EVID / "mapreduce_resultado.tsv").write_text(resultado, encoding="utf-8")
    (EVID / "hdfs_ls_raw.txt").write_text(sh("hdfs dfs -ls -R /datalake/raw", quiet=True).stdout, encoding="utf-8")
    print("modalidad\tcontratos\tvalor_total\n" + resultado)

correr_mapreduce()


FASE 4 - MAPREDUCE (Hadoop Streaming)
Map input records=10000
Map output records=10000
Map input records=10000
Map output records=10000
Map input records=10000
Map output records=10000
Map input records=10000
Map output records=10000
Map input records=10000
Map output records=10000
Map input records=5963
Map output records=5963
Reduce output records=13
2026-11-24 21:50:43,796 INFO mapreduce.Job: Job job_local936422038_0001 completed successfully
Map input records=55963
Map output records=55963
Reduce output records=13
modalidad	contratos	valor_total
Concurso de méritos abierto	114	83834448364
Contratación Directa (con ofertas)	1085	1823478398194
Contratación directa	43328	5311546045103
Contratación régimen especial	3898	874966198634
Contratación régimen especial (con ofertas)	984	1482462169638
Enajenación de bienes con sobre cerrado	1	1485000000
Enajenación de bienes con subasta	5	781818626
Licitación pública	117	697811917407
Licitación pública Obra Publica	78	420419567228
Mínima cuan

### Comparación MapReduce vs Spark
Spark hace exactamente el mismo cálculo, con las mismas reglas y sobre los mismos archivos raw, y se comparan los dos resultados fila a fila.

In [11]:
def comparar_mr_spark():
    titulo("FASE 4 - COMPARACIÓN MAPREDUCE vs SPARK")
    raw = leer_raw().cache()
    modalidad = F.trim(F.col("payload.modalidad_de_contratacion"))
    clave = F.when(modalidad.isNull() | (modalidad == ""), F.lit("SIN_DATO")).otherwise(modalidad)
    clave = F.when(F.col("_corrupt_record").isNotNull(), F.lit("__CORRUPTO__")).otherwise(clave)
    valor = F.coalesce(F.round(F.trim(F.col("payload.valor_del_contrato")).cast("decimal(38,4)"), 0).cast("long"), F.lit(0))
    spark_res = (raw.select(clave.alias("modalidad"), valor.alias("valor")).groupBy("modalidad")
                 .agg(F.count("*").alias("contratos_spark"), F.sum("valor").alias("valor_spark")))
    mr = (spark.read.option("sep", "\t").csv(f"{HDFS_URI}{HDFS_MR_OUT}")
          .toDF("modalidad", "contratos_mr", "valor_mr")
          .withColumn("contratos_mr", F.col("contratos_mr").cast("long"))
          .withColumn("valor_mr", F.col("valor_mr").cast("long")))
    comp = (mr.join(spark_res, "modalidad", "full_outer")
            .withColumn("coincide", (F.col("contratos_mr") == F.col("contratos_spark")) & (F.col("valor_mr") == F.col("valor_spark")))
            .orderBy(F.desc("contratos_spark")).toPandas())
    comp["coincide"] = comp["coincide"].fillna(False)
    comp.to_csv(EVID / "mapreduce_vs_spark.csv", index=False)
    raw.unpersist()
    print(f"Total raw -> MapReduce: {comp['contratos_mr'].sum():,} | Spark: {comp['contratos_spark'].sum():,}")
    print("¿Coinciden todas las modalidades (conteo y valor)?", "SÍ" if comp["coincide"].all() else "NO")
    return comp

comparar_mr_spark()


FASE 4 - COMPARACIÓN MAPREDUCE vs SPARK
Total raw -> MapReduce: 55,963 | Spark: 55,963
¿Coinciden todas las modalidades (conteo y valor)? SÍ


,modalidad,contratos_mr,valor_mr,contratos_spark,valor_spark,coincide
0,Contratación directa,43328,5311546045103,43328,5311546045103,True
1,Mínima cuantía,4944,219193902867,4944,219193902867,True
2,Contratación régimen especial,3898,874966198634,3898,874966198634,True
3,Contratación Directa (con ofertas),1085,1823478398194,1085,1823478398194,True
4,Contratación régimen especial (con ofertas),984,1482462169638,984,1482462169638,True
5,Selección abreviada subasta inversa,807,487973314407,807,487973314407,True
6,Selección Abreviada de Menor Cuantía,596,224731298741,596,224731298741,True
7,Licitación pública,117,697811917407,117,697811917407,True
8,Concurso de méritos abierto,114,83834448364,114,83834448364,True
9,Licitación pública Obra Publica,78,420419567228,78,420419567228,True


## Fases 5 y 6 · Zona curada con PySpark, calidad e incremental
| Problema | Decisión | Por qué |
|---|---|---|
| Líneas JSON corruptas | Se descartan | No hay datos recuperables |
| Sin `id_registro` | Se descartan | No se pueden deduplicar ni rastrear |
| Duplicados por `id_registro` | Se conserva la versión más reciente (`ultima_actualizacion`, luego `fecha_ingesta`) | Vienen del traslape de la marca de agua y de los reenvíos de Kafka |
| Texto inconsistente (`terminado`/`Terminado`, espacios, `No definido`, `Sin Descripcion`) | trim, un solo espacio, formato Título; los marcadores pasan a nulo | Así una categoría no se cuenta dos veces |
| Tipos: todo llega como texto | Fechas a timestamp/date, valores a `decimal(20,2)`, días a int | Permite sumar, filtrar y particionar |
| Sin `fecha_de_firma` | Se descarta | Es la columna de tiempo del análisis |
| Fecha fuera de [2015, hoy] | Se descarta | SECOP II existe desde 2015; una fecha futura es un error |
| Valor nulo | **Se conserva como nulo** | Imputar 0 bajaría los promedios; el contrato cuenta en los conteos, no en las sumas |
| Valor negativo | Se descarta | Un contrato no puede valer menos de cero |

**Partición por `anio_mes`:** la pregunta y el tablero son mensuales, y la carga incremental solo reescribe los meses afectados. Se descartaron `ciudad` (125 municipios muy desiguales, con Medellín concentrando la mayoría, lo que deja muchos archivos diminutos) y la fecha diaria (cientos de particiones pequeñas).

**Incremental:** solo se procesan los archivos raw de HDFS que no se han curado antes. Se unen con lo ya curado de esos meses, se deduplican y se reemplazan **solo esas particiones** (sobrescritura dinámica, pasando por una carpeta *staging*).

**Calidad:** antes de escribir corren 7 validaciones automáticas, 5 críticas y 2 de advertencia. Si una crítica falla, el pipeline se detiene y la zona curada no se toca.

In [12]:
class CalidadCriticaError(Exception):
    pass

def validar(df, run_id, registros_entrada, descartes):
    n = df.count()
    ahora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ids = df.select("id_registro").distinct().count()
    nulos_clave = df.filter(F.col("id_registro").isNull() | F.col("fecha_de_firma").isNull() | F.col("anio_mes").isNull()).count()
    fuera = df.filter((F.col("fecha_de_firma") < F.lit("2015-01-01").cast("timestamp")) |
                      (F.col("fecha_de_firma") > F.lit(ahora).cast("timestamp"))).count()
    negativos = df.filter(F.col("valor_del_contrato") < 0).count()
    valor_nulo = df.filter(F.col("valor_del_contrato").isNull()).count()
    otro_depto = df.filter(F.col("departamento") != DEPARTAMENTO).count()
    reglas = [
        ("conteo_mayor_que_cero", "CRITICA", n > 0, n),
        ("sin_nulos_en_columnas_clave (id, fecha_de_firma, anio_mes)", "CRITICA", nulos_clave == 0, nulos_clave),
        ("id_registro_sin_repetidos", "CRITICA", n == ids, n - ids),
        ("fecha_de_firma_en_rango [2015-01-01, hoy]", "CRITICA", fuera == 0, fuera),
        ("valor_del_contrato_no_negativo", "CRITICA", negativos == 0, negativos),
        ("valor_del_contrato_nulo_menor_5pct", "ADVERTENCIA", n == 0 or valor_nulo / n < 0.05, round(valor_nulo / n * 100, 2) if n else 0),
        (f"departamento_es_{DEPARTAMENTO}", "ADVERTENCIA", otro_depto == 0, otro_depto),
    ]
    resultado = [{"validacion": v, "nivel": nv, "ok": bool(ok), "valor_observado": obs} for v, nv, ok, obs in reglas]
    fallidas = [r for r in resultado if r["nivel"] == "CRITICA" and not r["ok"]]
    reporte = {"run_id": run_id, "fecha": ahora_iso(), "registros_entrada_raw": registros_entrada,
               "registros_validados": n,
               "registros_descartados": {"total": sum(descartes.values()), "por_motivo": descartes},
               "validaciones": resultado, "estado": "FALLIDO" if fallidas else "OK"}
    escribir_json(EVID / "calidad" / f"reporte_calidad_{run_id}.json", reporte)
    escribir_json(EVID / "calidad" / "reporte_calidad_ultimo.json", reporte)
    hist = EVID / "calidad" / "historial_calidad.csv"
    nuevo = not hist.exists()
    with open(hist, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["run_id", "fecha", "entrada_raw", "validados", "descartados", "estado"])
        w.writerow([run_id, reporte["fecha"], registros_entrada, n, reporte["registros_descartados"]["total"], reporte["estado"]])
    print("\nVALIDACIONES DE CALIDAD")
    for r in resultado:
        print(f"  [{'OK   ' if r['ok'] else 'FALLA'}] ({r['nivel']}) {r['validacion']}: {r['valor_observado']}")
    print(f"  Descartados: {reporte['registros_descartados']}")
    if fallidas:
        raise CalidadCriticaError("Validación crítica fallida: " + ", ".join(r["validacion"] for r in fallidas)
                                  + ". El pipeline se detiene y la zona curada NO se modifica.")
    return reporte

In [13]:
PLACEHOLDERS = ["", "no definido", "no definida", "sin descripcion", "sin descripción", "no aplica", "n/a", "na", "null", "none", "-"]
TEXTOS = ["nombre_entidad", "ciudad", "departamento", "orden", "sector", "rama", "entidad_centralizada",
          "estado_contrato", "tipo_de_contrato", "modalidad_de_contratacion", "justificacion_modalidad_de",
          "tipodocproveedor", "es_pyme", "origen_de_los_recursos", "destino_gasto", "objeto_del_contrato",
          "duraci_n_del_contrato", "referencia_del_contrato", "codigo_de_categoria_principal"]
CATEGORIAS = ["ciudad", "departamento", "estado_contrato", "tipo_de_contrato", "modalidad_de_contratacion", "sector", "orden", "rama"]
VALORES = ["valor_del_contrato", "valor_facturado", "valor_pagado"]

def dedup(df):
    w = Window.partitionBy("id_registro").orderBy(F.col("ultima_actualizacion").desc_nulls_last(),
                                                  F.col("fecha_ingesta").desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def curar(completo=False, simular_fallo=False):
    titulo("FASES 5/6 - ZONA CURADA (PySpark -> Parquet) + CALIDAD")
    run_id = datetime.now().strftime("%Y%m%dT%H%M%S")
    todos = hdfs_listar(HDFS_RAW)
    estado = leer_json(CURADA_ESTADO, {"archivos": []})
    nuevos = todos if completo else [a for a in todos if a not in set(estado["archivos"])]
    print(f"Archivos raw en HDFS: {len(todos)} | ya procesados: {len(estado['archivos'])} | por procesar: {len(nuevos)}")
    if not nuevos:
        print("No hay archivos raw nuevos: la zona curada no cambia (idempotente).")
        return None
    pasos = []
    def registrar(nombre, antes, df, decision):
        despues = df.count()
        pasos.append({"paso": nombre, "filas_antes": antes, "filas_despues": despues,
                      "descartadas": antes - despues, "decision": decision})
        print(f"  {nombre:<40} {antes:>9,} -> {despues:>9,}  (-{antes - despues:,})")
        return despues

    raw = leer_raw(nuevos).cache()
    n_raw = raw.count()
    print(f"Registros leídos de la zona raw: {n_raw:,}\n")
    df = raw.filter(F.col("_corrupt_record").isNull())
    n = registrar("a. quitar JSON corruptos", n_raw, df, "Una línea que no es JSON válido no tiene datos recuperables.")
    df = (df.select("id_registro", F.col("fecha_ingesta").cast("timestamp").alias("fecha_ingesta"), "run_id", "payload.*")
            .filter(F.col("id_registro").isNotNull() & (F.trim("id_registro") != "")))
    n = registrar("b. quitar registros sin id_registro", n, df, "Sin id no se puede deduplicar ni rastrear el contrato.")
    df = dedup(df).cache()
    n = registrar("c. quitar duplicados por id_registro", n, df,
                  "Se conserva la versión con ultima_actualizacion más reciente (si empata, la última ingesta).")
    antes_cat = {c: df.select(c).distinct().count() for c in CATEGORIAS}
    for c in TEXTOS:
        limpio = F.regexp_replace(F.trim(F.col(c)), r"\s+", " ")
        df = df.withColumn(c, F.when(F.lower(limpio).isin(PLACEHOLDERS), None).otherwise(limpio))
    for c in CATEGORIAS:
        df = df.withColumn(c, F.initcap(F.lower(F.col(c))))
    df = df.withColumn("nombre_entidad", F.upper("nombre_entidad"))
    despues_cat = {c: df.select(c).distinct().count() for c in CATEGORIAS}
    n = registrar("d. normalizar formatos de texto", n, df,
                  "trim + un solo espacio; 'No definido'/'Sin Descripcion' -> nulo; categorías en formato Título.")
    invalidos = df.filter(F.col("valor_del_contrato").isNotNull() & F.col("valor_del_contrato").cast("decimal(20,2)").isNull()).count()
    df = (df.withColumn("fecha_de_firma", F.col("fecha_de_firma").cast("timestamp"))
            .withColumn("fecha_de_inicio_del_contrato", F.to_date(F.col("fecha_de_inicio_del_contrato").cast("timestamp")))
            .withColumn("fecha_de_fin_del_contrato", F.to_date(F.col("fecha_de_fin_del_contrato").cast("timestamp")))
            .withColumn("ultima_actualizacion", F.col("ultima_actualizacion").cast("timestamp"))
            .withColumn("dias_adicionados", F.col("dias_adicionados").cast("int")))
    for c in VALORES:
        df = df.withColumn(c, F.col(c).cast("decimal(20,2)"))
    n = registrar("e. corregir tipos (fechas, números)", n, df,
                  f"Texto -> timestamp/date/decimal(20,2). No numéricos -> nulo ({invalidos} en valor_del_contrato).")
    df = df.filter(F.col("fecha_de_firma").isNotNull())
    n = registrar("f1. quitar sin fecha_de_firma", n, df, "Es la columna de tiempo del análisis y de la marca de agua.")
    hoy = datetime.now().strftime("%Y-%m-%d 23:59:59")
    df = df.filter((F.col("fecha_de_firma") >= F.lit("2015-01-01").cast("timestamp")) & (F.col("fecha_de_firma") <= F.lit(hoy).cast("timestamp")))
    n = registrar("f2. quitar fecha_de_firma fuera de rango", n, df, "SECOP II existe desde 2015; una fecha futura es un error.")
    df = df.filter(F.col("valor_del_contrato").isNull() | (F.col("valor_del_contrato") >= 0))
    n = registrar("f3. quitar valor_del_contrato negativo", n, df,
                  "Un contrato no puede valer menos de cero. El valor NULO se conserva (imputar 0 bajaría los promedios).")
    df = (df.withColumn("anio", F.year("fecha_de_firma")).withColumn("mes", F.month("fecha_de_firma"))
            .withColumn("anio_mes", F.date_format("fecha_de_firma", "yyyy-MM"))
            .withColumn("duracion_dias", F.when(F.col("fecha_de_fin_del_contrato") >= F.col("fecha_de_inicio_del_contrato"),
                                                F.datediff("fecha_de_fin_del_contrato", "fecha_de_inicio_del_contrato")))
            .withColumn("es_contratacion_directa", F.lower(F.col("modalidad_de_contratacion")).contains("directa"))
            .withColumn("es_persona_natural", F.lower(F.col("tipodocproveedor")).rlike("c[eé]dula")))
    if simular_fallo:
        print("\n  ** simular_fallo=True: se inyectan 5 filas duplicadas **")
        df = df.unionByName(df.limit(5))
    nuevos_limpios = df.cache()
    n_nuevos = nuevos_limpios.count()
    meses = [r[0] for r in nuevos_limpios.select("anio_mes").distinct().collect()]
    ruta_curada = f"{HDFS_URI}{HDFS_CURADA}"
    if hdfs_existe(HDFS_CURADA):
        previos = spark.read.parquet(ruta_curada).filter(F.col("anio_mes").isin(meses))
        n_previos = previos.count()
        combinado = previos.unionByName(nuevos_limpios)
        if not simular_fallo:
            combinado = dedup(combinado)
    else:
        n_previos, combinado = 0, nuevos_limpios
    combinado = combinado.cache()
    registrar("g. unir con curada existente + dedup", n_previos + n_nuevos, combinado,
              f"Meses afectados: {len(meses)}. Un contrato ya curado se reemplaza por su versión más reciente.")
    descartes = {p["paso"]: p["descartadas"] for p in pasos if p["descartadas"] > 0}
    try:
        validar(combinado, run_id, n_raw, descartes)
    except CalidadCriticaError as e:
        print(f"\n*** PIPELINE DETENIDO *** {e}")
        for d in (raw, nuevos_limpios, combinado):
            d.unpersist()
        raise
    staging = f"{HDFS_URI}{HDFS_STAGING}"
    combinado.repartition("anio_mes").write.mode("overwrite").partitionBy("anio_mes").parquet(staging)
    spark.read.parquet(staging).write.mode("overwrite").partitionBy("anio_mes").parquet(ruta_curada)
    hdfs_borrar(HDFS_STAGING)
    for d in (raw, nuevos_limpios, combinado):
        d.unpersist()
    estado["archivos"] = sorted(set(estado["archivos"]) | set(nuevos))
    estado["actualizado"] = ahora_iso()
    escribir_json(CURADA_ESTADO, estado)

    pd.DataFrame(pasos).to_csv(EVID / "curada_pasos_ultimo.csv", index=False)
    formatos = pd.DataFrame([{"columna": c, "valores_distintos_antes": antes_cat[c], "valores_distintos_despues": despues_cat[c]} for c in CATEGORIAS])
    formatos.to_csv(EVID / "curada_formatos_ultimo.csv", index=False)
    curada = spark.read.parquet(ruta_curada)
    (EVID / "esquema_curado.txt").write_text(curada._jdf.schema().treeString(), encoding="utf-8")
    total = curada.count()
    raw_b, cur_b = hdfs_tamano(HDFS_RAW), hdfs_tamano(HDFS_CURADA)
    tamanos = {"fecha": ahora_iso(), "raw_jsonl_mb": round(raw_b / 1e6, 2), "curada_parquet_mb": round(cur_b / 1e6, 2),
               "reduccion_pct": round((1 - cur_b / raw_b) * 100, 1) if raw_b else None,
               "filas_curada_total": total, "particiones_anio_mes": curada.select("anio_mes").distinct().count(),
               "ids_repetidos_en_toda_la_curada": total - curada.select("id_registro").distinct().count()}
    escribir_json(EVID / "tamanos_jsonl_vs_parquet.json", tamanos)
    print(f"\nZona curada: {total:,} filas en {tamanos['particiones_anio_mes']} particiones anio_mes")
    print(f"Tamaño raw JSONL: {tamanos['raw_jsonl_mb']} MB | curada Parquet: {tamanos['curada_parquet_mb']} MB | reducción: {tamanos['reduccion_pct']}%")
    return pd.DataFrame(pasos), formatos

pasos_df, formatos_df = curar()
display(pasos_df)
display(formatos_df)


FASES 5/6 - ZONA CURADA (PySpark -> Parquet) + CALIDAD
Archivos raw en HDFS: 6 | ya procesados: 0 | por procesar: 6
Registros leídos de la zona raw: 55,963

  a. quitar JSON corruptos                    55,963 ->    55,963  (-0)
  b. quitar registros sin id_registro         55,963 ->    55,963  (-0)
  c. quitar duplicados por id_registro        55,963 ->    55,963  (-0)
  d. normalizar formatos de texto             55,963 ->    55,963  (-0)
  e. corregir tipos (fechas, números)         55,963 ->    55,963  (-0)
  f1. quitar sin fecha_de_firma               55,963 ->    55,963  (-0)
  f2. quitar fecha_de_firma fuera de rango    55,963 ->    55,963  (-0)
  f3. quitar valor_del_contrato negativo      55,963 ->    55,963  (-0)
  g. unir con curada existente + dedup        55,963 ->    55,963  (-0)

VALIDACIONES DE CALIDAD
  [OK   ] (CRITICA) conteo_mayor_que_cero: 55963
  [OK   ] (CRITICA) sin_nulos_en_columnas_clave (id, fecha_de_firma, anio_mes): 0
  [OK   ] (CRITICA) id_registro_sin_re

,paso,filas_antes,filas_despues,descartadas,decision
0,a. quitar JSON corruptos,55963,55963,0,Una línea que no es JSON válido no tiene datos...
1,b. quitar registros sin id_registro,55963,55963,0,Sin id no se puede deduplicar ni rastrear el c...
2,c. quitar duplicados por id_registro,55963,55963,0,Se conserva la versión con ultima_actualizacio...
3,d. normalizar formatos de texto,55963,55963,0,trim + un solo espacio; 'No definido'/'Sin Des...
4,"e. corregir tipos (fechas, números)",55963,55963,0,"Texto -> timestamp/date/decimal(20,2). No numé..."
5,f1. quitar sin fecha_de_firma,55963,55963,0,Es la columna de tiempo del análisis y de la m...
6,f2. quitar fecha_de_firma fuera de rango,55963,55963,0,SECOP II existe desde 2015; una fecha futura e...
7,f3. quitar valor_del_contrato negativo,55963,55963,0,Un contrato no puede valer menos de cero. El v...
8,g. unir con curada existente + dedup,55963,55963,0,Meses afectados: 9. Un contrato ya curado se r...


,columna,valores_distintos_antes,valores_distintos_despues
0,ciudad,113,113
1,departamento,1,1
2,estado_contrato,7,7
3,tipo_de_contrato,20,20
4,modalidad_de_contratacion,13,13
5,sector,21,21
6,orden,3,3
7,rama,4,4


In [14]:
print(json.dumps(leer_json(EVID / "calidad" / "reporte_calidad_ultimo.json", {}), indent=2, ensure_ascii=False))
print(json.dumps(leer_json(EVID / "tamanos_jsonl_vs_parquet.json", {}), indent=2, ensure_ascii=False))
print(sh(f"hdfs dfs -ls {HDFS_CURADA}", quiet=True).stdout)
print((EVID / "esquema_curado.txt").read_text())

{
  "run_id": "20261124T215540",
  "fecha": "2026-11-24T21:56:39+00:00",
  "registros_entrada_raw": 55963,
  "registros_validados": 55963,
  "registros_descartados": {
    "total": 0,
    "por_motivo": {}
  },
  "validaciones": [
    {
      "validacion": "conteo_mayor_que_cero",
      "nivel": "CRITICA",
      "ok": true,
      "valor_observado": 55963
    },
    {
      "validacion": "sin_nulos_en_columnas_clave (id, fecha_de_firma, anio_mes)",
      "nivel": "CRITICA",
      "ok": true,
      "valor_observado": 0
    },
    {
      "validacion": "id_registro_sin_repetidos",
      "nivel": "CRITICA",
      "ok": true,
      "valor_observado": 0
    },
    {
      "validacion": "fecha_de_firma_en_rango [2015-01-01, hoy]",
      "nivel": "CRITICA",
      "ok": true,
      "valor_observado": 0
    },
    {
      "validacion": "valor_del_contrato_no_negativo",
      "nivel": "CRITICA",
      "ok": true,
      "valor_observado": 0
    },
    {
      "validacion": "valor_del_contrato_nulo_

### Demostración: una validación crítica detiene el pipeline
Se reprocesa la zona raw inyectando 5 filas duplicadas. La validación `id_registro_sin_repetidos` falla, el pipeline se detiene y la zona curada **no cambia**: el conteo de filas es el mismo antes y después.

In [15]:
antes = spark.read.parquet(f"{HDFS_URI}{HDFS_CURADA}").count()
try:
    curar(completo=True, simular_fallo=True)
except CalidadCriticaError as e:
    print("\nExcepción capturada ->", e)
despues = spark.read.parquet(f"{HDFS_URI}{HDFS_CURADA}").count()
print(f"\nFilas en la zona curada antes: {antes:,} | después: {despues:,} -> {'sin cambios' if antes == despues else 'CAMBIÓ'}")


FASES 5/6 - ZONA CURADA (PySpark -> Parquet) + CALIDAD
Archivos raw en HDFS: 6 | ya procesados: 6 | por procesar: 6
Registros leídos de la zona raw: 55,963

  a. quitar JSON corruptos                    55,963 ->    55,963  (-0)
  b. quitar registros sin id_registro         55,963 ->    55,963  (-0)
  c. quitar duplicados por id_registro        55,963 ->    55,963  (-0)
  d. normalizar formatos de texto             55,963 ->    55,963  (-0)
  e. corregir tipos (fechas, números)         55,963 ->    55,963  (-0)
  f1. quitar sin fecha_de_firma               55,963 ->    55,963  (-0)
  f2. quitar fecha_de_firma fuera de rango    55,963 ->    55,963  (-0)
  f3. quitar valor_del_contrato negativo      55,963 ->    55,963  (-0)

  ** simular_fallo=True: se inyectan 5 filas duplicadas **
  g. unir con curada existente + dedup       111,931 ->   111,931  (-0)

VALIDACIONES DE CALIDAD
  [OK   ] (CRITICA) conteo_mayor_que_cero: 111931
  [OK   ] (CRITICA) sin_nulos_en_columnas_clave (id, fecha_

## Fase 7 · Streaming: capa rápida (Lambda)
Spark Structured Streaming lee el tópico de Kafka y calcula, por **ventana de 1 día** de `fecha_de_firma` (tiempo del evento), los contratos, los contratos únicos (`approx_count_distinct` del id), los de contratación directa y el valor total.
- **Watermark de 3 días:** Spark mantiene abierta una ventana hasta ver eventos 3 días más recientes; entonces la cierra, la emite (modo *append*) y libera su estado. Los eventos que lleguen más tarde se descartan.
- **Checkpoint en HDFS** (`/checkpoints/secop_stream_diario`): guarda los offsets de Kafka ya leídos y el estado de las ventanas. **Al reiniciar continúa donde iba**, sin reprocesar ni duplicar.
- **Sink:** `foreachBatch` escribe en SQLite con `INSERT OR REPLACE` por `ventana_inicio`, así que si un micro-lote se reintenta se reescribe la misma fila.
- **Unión con el batch:** la tabla de servicio `ind_serie_diaria` usa el batch hasta su fecha de corte y la capa rápida para los días posteriores.

In [16]:
TABLA_STREAM = "stream_contratos_diarios"

def conteos_stream():
    con = sqlite3.connect(SQLITE_PATH)
    try:
        filas, suma = con.execute(f"SELECT COUNT(*), COALESCE(SUM(contratos),0) FROM {TABLA_STREAM}").fetchone()
    except sqlite3.OperationalError:
        filas, suma = 0, 0
    con.close()
    return filas, suma

def escribir_microlote(batch_df, batch_id):
    pdf = (batch_df.select(F.date_format("ventana.start", "yyyy-MM-dd").alias("ventana_inicio"),
                           F.date_format("ventana.end", "yyyy-MM-dd").alias("ventana_fin"),
                           "contratos", "contratos_unicos", "contratos_directa",
                           F.col("valor_total").cast("double").alias("valor_total"))
           .toPandas().fillna({"contratos_directa": 0, "valor_total": 0.0}))
    con = sqlite3.connect(SQLITE_PATH)
    con.execute(f"""CREATE TABLE IF NOT EXISTS {TABLA_STREAM} (ventana_inicio TEXT PRIMARY KEY, ventana_fin TEXT,
                    contratos INTEGER, contratos_unicos INTEGER, contratos_directa INTEGER, valor_total REAL,
                    batch_id INTEGER, actualizado TEXT)""")
    con.execute("CREATE TABLE IF NOT EXISTS stream_lotes (batch_id INTEGER PRIMARY KEY, ventanas_emitidas INTEGER, procesado TEXT)")
    ts = ahora_iso()
    con.executemany(f"INSERT OR REPLACE INTO {TABLA_STREAM} VALUES (?,?,?,?,?,?,?,?)",
                    [(r.ventana_inicio, r.ventana_fin, int(r.contratos), int(r.contratos_unicos),
                      int(r.contratos_directa), float(r.valor_total), batch_id, ts) for r in pdf.itertuples()])
    con.execute("INSERT OR REPLACE INTO stream_lotes VALUES (?,?,?)", (batch_id, len(pdf), ts))
    con.commit(); con.close()
    print(f"   micro-lote {batch_id}: {len(pdf)} ventanas cerradas escritas en SQLite")

def correr_streaming(segundos=90, etiqueta="corrida"):
    titulo(f"FASE 7 - STRUCTURED STREAMING ({etiqueta})")
    v_antes, c_antes = conteos_stream()
    print(f"ANTES   -> ventanas en SQLite: {v_antes} | contratos acumulados: {c_antes:,}")
    kafka = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
             .option("subscribe", KAFKA_TOPIC).option("startingOffsets", "earliest")
             .option("maxOffsetsPerTrigger", 50000).load())
    s = F.from_json(F.col("value").cast("string"), SOBRE).alias("s")
    eventos = (kafka.select(s).select(
                   F.col("s.id_registro").alias("id_registro"),
                   F.col("s.payload.fecha_de_firma").cast("timestamp").alias("fecha_firma"),
                   F.col("s.payload.valor_del_contrato").cast("decimal(20,2)").alias("valor"),
                   F.lower(F.col("s.payload.modalidad_de_contratacion")).contains("directa").cast("int").alias("directa"))
               .filter(F.col("fecha_firma").isNotNull()))
    por_dia = (eventos.withWatermark("fecha_firma", "3 days")
               .groupBy(F.window("fecha_firma", "1 day").alias("ventana"))
               .agg(F.count("*").alias("contratos"), F.approx_count_distinct("id_registro").alias("contratos_unicos"),
                    F.sum("directa").alias("contratos_directa"), F.sum("valor").alias("valor_total")))
    q = (por_dia.writeStream.outputMode("append").foreachBatch(escribir_microlote)
         .option("checkpointLocation", f"{HDFS_URI}{HDFS_CHECKPOINT}")
         .trigger(processingTime="10 seconds").queryName("secop_ventanas_diarias").start())
    print(f"Flujo iniciado; corre {segundos} s ...")
    q.awaitTermination(segundos)
    progreso = q.recentProgress
    q.stop()
    leidos = sum(p.get("numInputRows", 0) for p in progreso)
    lotes = sum(1 for p in progreso if p.get("numInputRows", 0) > 0)
    v_desp, c_desp = conteos_stream()
    print(f"Mensajes de Kafka leídos en esta corrida: {leidos:,} (micro-lotes con datos: {lotes})")
    print(f"DESPUÉS -> ventanas en SQLite: {v_desp} | contratos acumulados: {c_desp:,}")
    if leidos == 0:
        print("Reinicio sin mensajes nuevos: el checkpoint evitó reprocesar; los conteos no cambian.")
    ruta = EVID / "streaming_reinicio.csv"
    nuevo = not ruta.exists()
    with open(ruta, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["etiqueta", "fecha", "mensajes_leidos", "microlotes_con_datos", "ventanas_antes",
                        "ventanas_despues", "contratos_antes", "contratos_despues"])
        w.writerow([etiqueta, ahora_iso(), leidos, lotes, v_antes, v_desp, c_antes, c_desp])

correr_streaming(90, "streaming_corrida_1")


FASE 7 - STRUCTURED STREAMING (streaming_corrida_1)
ANTES   -> ventanas en SQLite: 0 | contratos acumulados: 0
Flujo iniciado; corre 90 s ...
   micro-lote 0: 0 ventanas cerradas escritas en SQLite
   micro-lote 1: 223 ventanas cerradas escritas en SQLite
   micro-lote 2: 29 ventanas cerradas escritas en SQLite
Mensajes de Kafka leídos en esta corrida: 55,963 (micro-lotes con datos: 2)
DESPUÉS -> ventanas en SQLite: 252 | contratos acumulados: 55,406


In [17]:
# REINICIO del flujo con el mismo checkpoint: no debe leer mensajes ni cambiar los conteos
correr_streaming(40, "streaming_reinicio")
display(pd.read_csv(EVID / "streaming_reinicio.csv"))
print(sh(f"hdfs dfs -ls -R {HDFS_CHECKPOINT}", quiet=True).stdout[:3000])


FASE 7 - STRUCTURED STREAMING (streaming_reinicio)
ANTES   -> ventanas en SQLite: 252 | contratos acumulados: 55,406
Flujo iniciado; corre 40 s ...
Mensajes de Kafka leídos en esta corrida: 0 (micro-lotes con datos: 0)
DESPUÉS -> ventanas en SQLite: 252 | contratos acumulados: 55,406
Reinicio sin mensajes nuevos: el checkpoint evitó reprocesar; los conteos no cambian.


,etiqueta,fecha,mensajes_leidos,microlotes_con_datos,ventanas_antes,ventanas_despues,contratos_antes,contratos_despues
0,streaming_corrida_1,2026-11-24T22:01:54+00:00,55963,2,0,252,0,55406
1,streaming_reinicio,2026-11-24T22:02:46+00:00,0,0,252,252,55406,55406


drwxr-xr-x   - root supergroup          0 2026-11-24 22:00 /checkpoints/secop_stream_diario/commits
-rw-r--r--   1 root supergroup         41 2026-11-24 22:00 /checkpoints/secop_stream_diario/commits/0
-rw-r--r--   1 root supergroup         41 2026-11-24 22:00 /checkpoints/secop_stream_diario/commits/1
-rw-r--r--   1 root supergroup         41 2026-11-24 22:00 /checkpoints/secop_stream_diario/commits/2
-rw-r--r--   1 root supergroup         45 2026-11-24 22:00 /checkpoints/secop_stream_diario/metadata
drwxr-xr-x   - root supergroup          0 2026-11-24 22:00 /checkpoints/secop_stream_diario/offsets
-rw-r--r--   1 root supergroup        817 2026-11-24 22:00 /checkpoints/secop_stream_diario/offsets/0
-rw-r--r--   1 root supergroup        829 2026-11-24 22:00 /checkpoints/secop_stream_diario/offsets/1
-rw-r--r--   1 root supergroup        829 2026-11-24 22:00 /checkpoints/secop_stream_diario/offsets/2
drwxr-xr-x   - root supergroup          0 2026-11-24 22:00 /checkpoints/secop_stream_di

## Fase 8 · Zona de servicio (SQLite)
Los indicadores se calculan **una sola vez** y se guardan listos en SQLite, de modo que el tablero solo lee y grafica. Las tablas se reconstruyen completas en cada corrida: son agregados pequeños, y correr dos veces deja el mismo resultado.

| Tabla | Contenido |
|---|---|
| `ind_resumen` | KPIs: contratos, valor total, % de contratación directa, entidad líder y su % del valor |
| `ind_mensual` | Contratos, valor y % de contratación directa por mes |
| `ind_top_entidades` | Las 15 entidades con mayor valor contratado |
| `ind_modalidad` | % de los contratos y % del valor por modalidad |
| `ind_municipios` | Los 15 municipios con más contratos |
| `ind_serie_diaria` | Serie diaria Lambda (batch + streaming) |
| `metadata` | Fuente, dataset, filtro y fecha de corte |

In [18]:
def pct(num, den):
    return F.round(F.when(den > 0, num / den * 100), 2)

def construir_servicio():
    titulo("FASE 8 - ZONA DE SERVICIO (indicadores -> SQLite)")
    df = (spark.read.parquet(f"{HDFS_URI}{HDFS_CURADA}")
          .withColumn("valor", F.col("valor_del_contrato").cast("double"))
          .withColumn("directa", F.col("es_contratacion_directa").cast("int")).cache())
    total = df.count()
    valor_total = df.agg(F.sum("valor")).first()[0] or 0.0
    fecha_corte = df.agg(F.max("fecha_de_firma")).first()[0]
    fecha_min = df.agg(F.min("fecha_de_firma")).first()[0]
    t = {}
    t["ind_mensual"] = (df.groupBy("anio_mes").agg(
            F.count("*").alias("contratos"), F.sum("valor").alias("valor_total"), F.avg("valor").alias("valor_promedio"),
            F.sum("directa").alias("contratos_directa"),
            F.sum(F.when(F.col("directa") == 1, F.col("valor"))).alias("valor_directa"))
        .withColumn("pct_contratos_directa", pct(F.col("contratos_directa"), F.col("contratos")))
        .withColumn("pct_valor_directa", pct(F.col("valor_directa"), F.col("valor_total")))
        .orderBy("anio_mes").toPandas())
    t["ind_top_entidades"] = (df.groupBy("nombre_entidad").agg(
            F.count("*").alias("contratos"), F.sum("valor").alias("valor_total"), F.sum("directa").alias("contratos_directa"))
        .withColumn("pct_valor_total", pct(F.col("valor_total"), F.lit(valor_total)))
        .withColumn("pct_contratos_directa", pct(F.col("contratos_directa"), F.col("contratos")))
        .orderBy(F.desc("valor_total")).limit(15).toPandas())
    t["ind_modalidad"] = (df.groupBy(F.coalesce("modalidad_de_contratacion", F.lit("Sin dato")).alias("modalidad"))
        .agg(F.count("*").alias("contratos"), F.sum("valor").alias("valor_total"))
        .withColumn("pct_contratos", pct(F.col("contratos"), F.lit(total)))
        .withColumn("pct_valor", pct(F.col("valor_total"), F.lit(valor_total)))
        .orderBy(F.desc("contratos")).toPandas())
    t["ind_municipios"] = (df.groupBy(F.coalesce("ciudad", F.lit("Sin dato")).alias("municipio"))
        .agg(F.count("*").alias("contratos"), F.sum("valor").alias("valor_total"))
        .orderBy(F.desc("contratos")).limit(15).toPandas())
    serie = (df.groupBy(F.date_format("fecha_de_firma", "yyyy-MM-dd").alias("fecha"))
             .agg(F.count("*").alias("contratos"), F.sum("valor").alias("valor_total"))
             .withColumn("origen", F.lit("batch")).orderBy("fecha").toPandas())
    con = sqlite3.connect(SQLITE_PATH)
    try:   # unión Lambda: batch hasta su fecha de corte + capa rápida después
        st = pd.read_sql(f"SELECT ventana_inicio AS fecha, contratos_unicos AS contratos, valor_total FROM {TABLA_STREAM}", con)
        st = st[st["fecha"] > serie["fecha"].max()].assign(origen="streaming")
        print(f"Serie diaria: {len(serie)} días batch + {len(st)} días de la capa rápida posteriores al corte batch")
        serie = pd.concat([serie, st], ignore_index=True).sort_values("fecha")
    except Exception:
        print("Aún no hay tabla de streaming: la serie diaria usa solo el batch.")
    t["ind_serie_diaria"] = serie
    directa = int(df.agg(F.sum("directa")).first()[0] or 0)
    top = t["ind_top_entidades"].iloc[0]
    t["ind_resumen"] = pd.DataFrame([{
        "contratos": total, "valor_total": valor_total, "valor_promedio": valor_total / total if total else 0,
        "entidades": df.select("nombre_entidad").distinct().count(),
        "municipios": df.select("ciudad").distinct().count(),
        "pct_contratos_directa": round(directa / total * 100, 2) if total else 0,
        "top_entidad": top["nombre_entidad"], "pct_valor_top_entidad": float(top["pct_valor_total"]),
        "pct_valor_top5_entidades": float(t["ind_top_entidades"]["pct_valor_total"].head(5).sum())}])
    t["metadata"] = pd.DataFrame([{
        "fuente": "SECOP II - Contratos Electrónicos (Colombia Compra Eficiente)", "portal": "datos.gov.co",
        "dataset": DATASET_ID, "url_api": API_URL, "filtro": f"departamento = {DEPARTAMENTO}",
        "fecha_min_firma": str(fecha_min), "fecha_corte": str(fecha_corte), "actualizado": ahora_iso()}])
    for nombre, pdf in t.items():
        pdf.to_sql(nombre, con, if_exists="replace", index=False)
        pdf.to_csv(SERVICIO_DIR / f"{nombre}.csv", index=False)
        print(f"  {nombre:<20} {len(pdf):>5} filas")
    con.commit(); con.close()
    df.unpersist()
    r = t["ind_resumen"].iloc[0]
    print(f"\nContratos: {r['contratos']:,} | Valor total: ${r['valor_total']:,.0f} COP | "
          f"% directa: {r['pct_contratos_directa']}% | Entidad líder: {r['top_entidad']} ({r['pct_valor_top_entidad']}% del valor)")
    print(f"Fecha de corte: {fecha_corte}")
    return t

servicio = construir_servicio()
display(servicio["ind_resumen"]); display(servicio["ind_mensual"]); display(servicio["ind_top_entidades"].head(10))


FASE 8 - ZONA DE SERVICIO (indicadores -> SQLite)
Serie diaria: 248 días batch + 0 días de la capa rápida posteriores al corte batch
  ind_mensual              9 filas
  ind_top_entidades       15 filas
  ind_modalidad           13 filas
  ind_municipios          15 filas
  ind_serie_diaria       248 filas
  ind_resumen              1 filas
  metadata                 1 filas

Contratos: 55,963 | Valor total: $11,635,090,043,715 COP | % directa: 79.36% | Entidad líder: DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNOVACION DE MEDELLIN (30.11% del valor)
Fecha de corte: 2026-11-22 14:06:35.779000


,contratos,valor_total,valor_promedio,entidades,municipios,pct_contratos_directa,top_entidad,pct_valor_top_entidad,pct_valor_top5_entidades
0,55963,1.163509e+13,2.079068e+08,485,113,79.36,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,30.11,56.8


,anio_mes,contratos,valor_total,valor_promedio,contratos_directa,valor_directa,pct_contratos_directa,pct_valor_directa
0,2026-03,1732,4.419641e+11,2.551756e+08,210,4.792909e+10,12.12,10.84
1,2026-04,1359,4.514036e+11,3.321586e+08,186,2.555435e+10,13.69,5.66
2,2026-05,1360,3.686039e+11,2.710323e+08,141,2.990344e+10,10.37,8.11
3,2026-06,2204,8.292672e+11,3.762556e+08,1044,2.751454e+11,47.37,33.18
4,2026-07,15768,1.747813e+12,1.108456e+08,14314,1.221668e+12,90.78,69.90
5,2026-08,11910,1.489398e+12,1.250544e+08,10666,7.980642e+11,89.55,53.58
6,2026-09,9651,2.681286e+12,2.778247e+08,8180,2.065923e+12,84.76,77.05
7,2026-10,7958,2.488076e+12,3.126510e+08,6524,1.927121e+12,81.98,77.45
8,2026-11,4021,1.137277e+12,2.828344e+08,3148,7.437163e+11,78.29,65.39


,nombre_entidad,contratos,valor_total,contratos_directa,pct_valor_total,pct_contratos_directa
0,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,645,3.503454e+12,412,30.11,63.88
1,INSTITUTO DE DEPORTES Y RECREACION DE MEDELLIN,4163,1.602407e+12,4030,13.77,96.81
2,DEPARTAMENTO DE ANTIOQUIA//,665,6.936179e+11,525,5.96,78.95
3,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,547,5.935066e+11,99,5.10,18.10
4,MUNICIPIO DE ENVIGADO*,472,2.166589e+11,364,1.86,77.12
5,INSTITUCIÓN UNIVERSITARIA ITM,10140,2.088164e+11,10021,1.79,98.83
6,AREA METROPOLITANA DEL VALLE DE ABURRA,136,2.003320e+11,88,1.72,64.71
7,ICBF REGIONAL ANTIOQUIA,917,1.814103e+11,623,1.56,67.94
8,RENTING DE ANTIOQUIA,11,1.792285e+11,0,1.54,0.00
9,EMPRESA DE VIVIENDA DE ANTIOQUIA-VIVA,14,1.555556e+11,0,1.34,0.00


## Fase 8 · Tablero (Plotly)
El tablero lee **solo** de SQLite, sin hacer cálculos de negocio. Tiene 5 KPIs y 5 visualizaciones. El título responde la pregunta con los datos, y el pie de página indica la fuente y la fecha de corte. Se guarda como `dashboard/tablero.html`, un archivo autocontenido que se abre con doble clic.

In [19]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

AZUL, NARANJA, GRIS = "#1f4e79", "#e07b24", "#8c96a3"
MM = 1e9

def fmt_cop(v):
    return f"${v / 1e12:,.2f} billones" if v >= 1e12 else f"${v / MM:,.1f} mil millones"

def estilo(fig, alto=420):
    fig.update_layout(template="plotly_white", height=alto, margin=dict(l=10, r=10, t=50, b=10),
                      font=dict(family="Segoe UI, Arial, sans-serif", size=13),
                      legend=dict(orientation="h", y=-0.15), title_font_size=16)
    return fig

def construir_tablero(mostrar=True):
    titulo("FASE 8 - TABLERO (SQLite -> Plotly HTML)")
    con = sqlite3.connect(SQLITE_PATH)
    t = {n: pd.read_sql(f"SELECT * FROM {n}", con) for n in
         ["metadata", "ind_resumen", "ind_mensual", "ind_top_entidades", "ind_modalidad", "ind_municipios", "ind_serie_diaria"]}
    con.close()
    meta, r = t["metadata"].iloc[0], t["ind_resumen"].iloc[0]
    m = t["ind_mensual"]
    f1 = make_subplots(specs=[[{"secondary_y": True}]])
    f1.add_bar(x=m["anio_mes"], y=m["valor_total"] / MM, name="Valor contratado (mil millones COP)", marker_color=AZUL,
               customdata=m["contratos"], hovertemplate="%{x}<br>$%{y:,.1f} mil millones<br>%{customdata:,} contratos<extra></extra>")
    f1.add_scatter(x=m["anio_mes"], y=m["pct_contratos_directa"], name="% contratos por contratación directa",
                   mode="lines+markers", line=dict(color=NARANJA, width=3), secondary_y=True,
                   hovertemplate="%{x}<br>%{y:.1f}% directa<extra></extra>")
    f1.update_yaxes(title_text="Mil millones COP", secondary_y=False)
    f1.update_yaxes(title_text="% contratación directa", range=[0, 100], secondary_y=True)
    estilo(f1).update_layout(title="1. Valor contratado por mes y peso de la contratación directa")
    e = t["ind_top_entidades"].head(10).iloc[::-1]
    f2 = go.Figure(go.Bar(x=e["valor_total"] / MM, y=e["nombre_entidad"].str.slice(0, 45), orientation="h", marker_color=AZUL,
                          text=e["pct_valor_total"].map(lambda v: f"{v:.1f}%"), textposition="outside",
                          customdata=e[["contratos", "pct_contratos_directa"]],
                          hovertemplate="%{y}<br>$%{x:,.1f} mil millones<br>%{customdata[0]:,} contratos<br>%{customdata[1]:.1f}% por directa<extra></extra>"))
    estilo(f2, 460).update_layout(title="2. Las 10 entidades con mayor valor contratado (% del total)", xaxis_title="Mil millones COP")
    mo = t["ind_modalidad"].head(8).iloc[::-1]
    f3 = go.Figure()
    f3.add_bar(y=mo["modalidad"], x=mo["pct_contratos"], name="% de los contratos", orientation="h", marker_color=NARANJA)
    f3.add_bar(y=mo["modalidad"], x=mo["pct_valor"], name="% del valor", orientation="h", marker_color=AZUL)
    estilo(f3, 460).update_layout(title="3. Modalidad de contratación: cantidad vs. valor", barmode="group", xaxis_title="%")
    mu = t["ind_municipios"].head(10).iloc[::-1]
    f4 = go.Figure(go.Bar(x=mu["contratos"], y=mu["municipio"], orientation="h", marker_color=GRIS,
                          hovertemplate="%{y}<br>%{x:,} contratos<extra></extra>"))
    estilo(f4).update_layout(title="4. Municipios con más contratos firmados", xaxis_title="Contratos")
    s = t["ind_serie_diaria"]
    f5 = go.Figure()
    for origen, color in (("batch", AZUL), ("streaming", NARANJA)):
        sub = s[s["origen"] == origen]
        if len(sub):
            f5.add_scatter(x=sub["fecha"], y=sub["contratos"], mode="lines", name=f"Capa {origen}", line=dict(color=color, width=1.5))
    estilo(f5, 360).update_layout(title="5. Contratos firmados por día (Lambda: batch + streaming)", yaxis_title="Contratos")

    titulo_t = (f"En Antioquia, {r['pct_contratos_directa']:.0f}% de los contratos públicos se firman por contratación directa, "
                f"y {str(r['top_entidad']).title()} concentra el {r['pct_valor_top_entidad']:.0f}% del valor contratado")
    kpis = [("Contratos analizados", f"{int(r['contratos']):,}".replace(",", ".")),
            ("Valor total contratado", fmt_cop(r["valor_total"])),
            ("Contratos por contratación directa", f"{r['pct_contratos_directa']:.1f}%"),
            ("Valor en las 5 entidades principales", f"{r['pct_valor_top5_entidades']:.1f}%"),
            ("Entidades contratantes", f"{int(r['entidades']):,}".replace(",", "."))]
    tarjetas = "".join(f'<div class="kpi"><div class="v">{v}</div><div class="l">{l}</div></div>' for l, v in kpis)
    figs = [f1, f2, f3, f4, f5]
    g = [f.to_html(full_html=False, include_plotlyjs=(i == 0), config={"displaylogo": False}) for i, f in enumerate(figs)]
    html = f"""<!DOCTYPE html><html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1"><title>Contratación pública en Antioquia - SECOP II</title>
<style>
 body {{ font-family:'Segoe UI',Arial,sans-serif; margin:0; background:#f4f6f9; color:#1d2733; }}
 header {{ background:{AZUL}; color:white; padding:28px 40px; }}
 header .p {{ font-size:14px; opacity:.85; margin:0 0 6px; text-transform:uppercase; letter-spacing:.5px; }}
 header h1 {{ margin:0; font-size:26px; line-height:1.3; font-weight:600; max-width:1100px; }}
 main {{ padding:24px 40px; max-width:1400px; margin:auto; }}
 .kpis {{ display:grid; grid-template-columns:repeat(5,1fr); gap:14px; margin-bottom:20px; }}
 .kpi {{ background:white; border-radius:10px; padding:16px 18px; box-shadow:0 1px 3px rgba(0,0,0,.08); }}
 .kpi .v {{ font-size:24px; font-weight:700; color:{AZUL}; }} .kpi .l {{ font-size:13px; color:#5b6673; margin-top:4px; }}
 .grid {{ display:grid; grid-template-columns:1fr 1fr; gap:16px; }}
 .card {{ background:white; border-radius:10px; padding:8px; box-shadow:0 1px 3px rgba(0,0,0,.08); }} .full {{ grid-column:1/-1; }}
 footer {{ padding:16px 40px 32px; font-size:13px; color:#5b6673; max-width:1400px; margin:auto; }}
 @media (max-width:900px) {{ .kpis {{ grid-template-columns:1fr 1fr; }} .grid {{ grid-template-columns:1fr; }} }}
</style></head><body>
<header><p class="p">¿Qué entidades concentran el valor contratado en Antioquia y qué peso tiene la contratación directa?</p>
<h1>{titulo_t}</h1></header>
<main><div class="kpis">{tarjetas}</div><div class="grid">
<div class="card full">{g[0]}</div><div class="card">{g[1]}</div><div class="card">{g[2]}</div>
<div class="card">{g[3]}</div><div class="card">{g[4]}</div></div></main>
<footer><b>Fuente:</b> {meta['fuente']} — {meta['portal']}, dataset <code>{meta['dataset']}</code> ({meta['url_api']}), filtro {meta['filtro']}.<br>
<b>Periodo:</b> contratos firmados entre {str(meta['fecha_min_firma'])[:10]} y <b>{str(meta['fecha_corte'])[:10]} (fecha de corte)</b>.
Tablero generado el {str(meta['actualizado'])[:16].replace('T', ' ')} desde la zona de servicio (SQLite). Valores en pesos colombianos corrientes.</footer>
</body></html>"""
    (DASH / "tablero.html").write_text(html, encoding="utf-8")
    print("Tablero guardado en dashboard/tablero.html")
    print("TÍTULO:", titulo_t)
    if mostrar:
        for f in figs:
            f.show()
    return titulo_t

titulo_tablero = construir_tablero()


FASE 8 - TABLERO (SQLite -> Plotly HTML)
Tablero guardado en dashboard/tablero.html
TÍTULO: En Antioquia, 79% de los contratos públicos se firman por contratación directa, y Distrito Especial De Ciencia Tecnologia E Innovacion De Medellin concentra el 30% del valor contratado


## Fase 6 · Idempotencia: el pipeline completo corre dos veces sin duplicar
`conteos()` cuenta los registros de cada zona y los guarda en `evidencias/idempotencia_conteos.csv`. Luego `pipeline_completo()` corre de nuevo **todas** las fases:
- la ingesta usa la marca de agua y solo trae lo nuevo, más el traslape de `>=`;
- el productor solo publica el archivo nuevo, y el consumidor continúa desde su offset;
- la zona curada deduplica por id y reemplaza solo los meses afectados;
- el streaming continúa desde su checkpoint.

**Resultado esperado:** en la corrida 2, `curada_filas == curada_ids_distintos`, y la curada solo crece si SECOP publicó contratos realmente nuevos.

In [20]:
def conteos(etiqueta):
    raw_local = 0
    for p in (RAW_DIR / FUENTE / DATASET_NOMBRE).glob("fecha_ingesta=*/*.jsonl"):
        with open(p, encoding="utf-8") as f:
            raw_local += sum(1 for _ in f)
    curada = spark.read.parquet(f"{HDFS_URI}{HDFS_CURADA}")
    con = sqlite3.connect(SQLITE_PATH)
    servicio_n = con.execute("SELECT COALESCE(SUM(contratos),0) FROM ind_mensual").fetchone()[0]
    con.close()
    fila = {"etiqueta": etiqueta, "fecha": ahora_iso(), "raw_local_lineas": raw_local,
            "raw_hdfs_registros": leer_raw().count(), "curada_filas": curada.count(),
            "curada_ids_distintos": curada.select("id_registro").distinct().count(),
            "servicio_contratos": servicio_n, "streaming_contratos": conteos_stream()[1]}
    ruta = EVID / "idempotencia_conteos.csv"
    nuevo = not ruta.exists()
    with open(ruta, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(fila))
        if nuevo:
            w.writeheader()
        w.writerow(fila)
    return fila

def pipeline_completo(etiqueta, segundos_streaming=40):
    t0 = time.time()
    ingestar(); producir(); consumir_a_hdfs(); curar()
    correr_streaming(segundos_streaming, etiqueta); construir_servicio(); construir_tablero(mostrar=False)
    print(f"\nPipeline completo en {(time.time() - t0) / 60:.1f} min")
    return conteos(etiqueta)

print(conteos("corrida_1"))
print(pipeline_completo("corrida_2"))
display(pd.read_csv(EVID / "idempotencia_conteos.csv"))

{'etiqueta': 'corrida_1', 'fecha': '2026-11-24T22:06:14+00:00', 'raw_local_lineas': 55963, 'raw_hdfs_registros': 55963, 'curada_filas': 55963, 'curada_ids_distintos': 55963, 'servicio_contratos': 55963, 'streaming_contratos': 55406}

FASE 2 - INGESTA API SECOP II -> ZONA RAW
run_id: 20261124T220618_a554d1 | marca de agua anterior: 2026-11-22T09:06:35.779
Consulta: departamento = 'Antioquia' AND fecha_de_firma >= '2026-11-22T09:06:35.779' AND fecha_de_firma <= '2026-11-24T23:59:59.999'


   página   1 | offset       0 | +    1 | acumulado       1

Registros: 1 (sin id: 0, reintentos: 0) | nueva marca de agua: 2026-11-22T09:06:35.779

FASE 3 - PRODUCTOR KAFKA (raw -> tópico, key = id_registro)
Tópico ya existe: secop_contratos
Particiones del tópico: 3
Archivos raw: 2 | ya publicados: 1 | a publicar: 1


/tmp/ipykernel_12638/4044530019.py:25: DeprecationWarning:

key_serializer does not implement kafka.serializer.Serializer

/tmp/ipykernel_12638/4044530019.py:25: DeprecationWarning:

value_serializer does not implement kafka.serializer.Serializer

/tmp/ipykernel_12638/1949313614.py:5: DeprecationWarning:

value_deserializer does not implement kafka.serializer.Deserializer



   part-20261124T220618_a554d1.jsonl: 1 mensajes
Total publicados: 1 | por partición: {2: 1}

FASE 3/4 - CONSUMIDOR KAFKA -> HDFS (un archivo por lote)
   lote 1:      1 mensajes -> /datalake/raw/datos.gov.co/secop_contratos/fecha_ingesta=2026-11-24/lote_20261124T220625_0001_35809e.jsonl
Mensajes consumidos: 1 en 1 lote(s) | por partición: {2: 1}

FASES 5/6 - ZONA CURADA (PySpark -> Parquet) + CALIDAD
Archivos raw en HDFS: 7 | ya procesados: 6 | por procesar: 1
Registros leídos de la zona raw: 1

  a. quitar JSON corruptos                         1 ->         1  (-0)
  b. quitar registros sin id_registro              1 ->         1  (-0)
  c. quitar duplicados por id_registro             1 ->         1  (-0)
  d. normalizar formatos de texto                  1 ->         1  (-0)
  e. corregir tipos (fechas, números)              1 ->         1  (-0)
  f1. quitar sin fecha_de_firma                    1 ->         1  (-0)
  f2. quitar fecha_de_firma fuera de rango         1 ->         1 

,etiqueta,fecha,raw_local_lineas,raw_hdfs_registros,curada_filas,curada_ids_distintos,servicio_contratos,streaming_contratos
0,corrida_1,2026-11-24T22:06:14+00:00,55963,55963,55963,55963,55963,55406
1,corrida_2,2026-11-24T22:07:45+00:00,55964,55964,55963,55963,55963,55406


## Evidencias de Kafka y HDFS
Se guardan en `evidencias/` las salidas de: descripción del tópico (3 particiones), grupo consumidor con sus offsets, `hdfs dfs -ls -R` del data lake, tamaño de cada zona y checkpoint del streaming.

In [21]:
evid_cmds = {
    "kafka_topico_describe.txt": f"kafka-topics.sh --bootstrap-server {KAFKA_BOOTSTRAP} --describe --topic {KAFKA_TOPIC}",
    "kafka_grupo_consumidor.txt": f"kafka-consumer-groups.sh --bootstrap-server {KAFKA_BOOTSTRAP} --describe --group {KAFKA_GRUPO}",
    "kafka_offsets_topico.txt": f"kafka-get-offsets.sh --bootstrap-server {KAFKA_BOOTSTRAP} --topic {KAFKA_TOPIC}",
    "hdfs_ls_datalake.txt": "hdfs dfs -ls -R /datalake",
    "hdfs_du_zonas.txt": "hdfs dfs -du -s -h /datalake/raw /datalake/curada",
    "hdfs_checkpoint_streaming.txt": f"hdfs dfs -ls -R {HDFS_CHECKPOINT}",
    "hdfs_report.txt": "hdfs dfsadmin -report",
}
for archivo, cmd in evid_cmds.items():
    r = sh(cmd, check=False, quiet=True)
    salida = (r.stdout + ("\n" + r.stderr if r.returncode else "")).strip()
    (EVID / archivo).write_text(f"$ {cmd}\n{salida}\n", encoding="utf-8")
    print(f"\n### {archivo}\n$ {cmd}\n{salida[:1500]}")


### kafka_topico_describe.txt
$ kafka-topics.sh --bootstrap-server localhost:9092 --describe --topic secop_contratos
Topic: secop_contratos	TopicId: zMcMLQ6XSaG0rTT-vbFFHA	PartitionCount: 3	ReplicationFactor: 1	Configs: segment.bytes=1073741824
	Topic: secop_contratos	Partition: 0	Leader: 1	Replicas: 1	Isr: 1
	Topic: secop_contratos	Partition: 1	Leader: 1	Replicas: 1	Isr: 1
	Topic: secop_contratos	Partition: 2	Leader: 1	Replicas: 1	Isr: 1

### kafka_grupo_consumidor.txt
$ kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group secop-hdfs-writer
GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
secop-hdfs-writer secop_contratos 0          18752           18752           0               -               -               -
secop-hdfs-writer secop_contratos 1          18611           18611           0               -               -               -
secop-hdfs-writer secop_contratos 2  

## Descargar resultados para el repositorio
Se genera `resultados_pipeline.zip` con las carpetas `evidencias/`, `data/` (manifiestos, muestra raw y tablas de servicio en SQLite y CSV), `dashboard/`, `estado/` y `mapreduce/`. El raw completo no se incluye porque es pesado.

Para el repositorio, descarguen además este notebook **ya ejecutado** con `Archivo → Descargar → Descargar .ipynb`.

In [22]:
import zipfile
zip_path = Path("/content/resultados_pipeline.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in BASE.rglob("*"):
        if p.is_file() and "data/raw" not in str(p.relative_to(BASE)).replace("\\", "/"):
            z.write(p, p.relative_to(BASE))
print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Descárguelo desde el panel de archivos (izquierda):", e)

/content/resultados_pipeline.zip (1.4 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>